## Bölüm 4 — Taslak raporun eleştirisi

| Taslaktaki iddia | Veride | Durum |
|---|---|---|
| Memnuniyette %21 eksik, ortalama 3,94 | %21, 3,94 | ✅ Doğru |
| Servis merkezi ortalamaları (3,91–3,99) | Aynı | ✅ Doğru (yorum yanlış) |
| 8.400 müşteri | 4.200 benzersiz | ❌ Uyuşmuyor |
| 32.770 satırlık müşteri seviyesinde tablo | 32.770 ham servis kaydı | ❌ Müşteri seviyesi değil |
| Terk edenlerin memnuniyeti 3,71, kalanların 3,98; p = 0,03 | 3,79'a karşı 4,01; p < 0,001 | ❌ Uyuşmuyor (yöntem belirtilmediği için kesin değil) |

### Ana hatalar

| # | Hata | Neden yanlış | Doğrusu |
|---|---|---|---|
| 1 | *"Modele mevcut tüm değişkenleri verdik; algoritma ilgisiz değişkenleri kendisi eleyecektir."* (ROC-AUC 0,99) | Algoritma tahmin gücüne bakar, bilginin tahmin anında var olup olmadığını bilemez. `crm_musteri_durumu` ("bugün itibarıyla" durum) ve `son_iletisim_notu` (tarihsiz) terk gerçekleştikten sonra yazılıyor: "Kaybedildi" ve "Başka servise geçti" gibi değerlerin %100'ü terk. Model terki tahmin etmiyor, okuyor. | Kesim tarihinde bilinmeyen alanlar modelden çıkarılmalı. Çıkarıldığında gerçekçi performans ROC-AUC ≈ 0,78. |
| 2 | *"%84 doğruluk... operasyonel kullanım için fazlasıyla yeterlidir."* | Terk oranı %16,6. Herkese "terk etmez" diyen, hiçbir şey öğrenmemiş bir model %83,4 doğruluk alır. %84, taban çizgisinin kendisi. | Dengesiz veride doğruluk yerine ROC-AUC, PR-AUC ve "en riskli %20'de yakalanan terk oranı" raporlanmalı ve basit bir kuralla (ör. yalnızca recency) karşılaştırılmalı. |
| 3 | *"`crm_musteri_durumu`'nun en güçlü değişken çıkması CRM verimizin kalitesini göstermektedir... CRM'in daha sık güncellenmesi model performansını artıracaktır."* | Değişkenin en üstte çıkması bir sızıntı işareti, kalite işareti değil. CRM'i daha sık güncellemek sızıntıyı artırır; terk anında bilinen bir bilgi üretmez. | Bu sonuç bir alarm olarak görülüp araştırılmalıydı. |
| 4 | *"Veri setinde toplam 8.400 müşteri bulunmaktadır."* | Veride 4.200 benzersiz müşteri var (4.235 satır, 35'i birebir kopya). 8.400'ü üreten makul bir birleştirme bulunamadı; tam iki katı olması çift sayım şüphesi doğuruyor. | 4.200 benzersiz müşteri; kesimden önce servisi olan popülasyon 3.974. |
| 5 | *"SM-Maslak... memnuniyet sıralamasının en altında... acil kalite denetimi... iş hacminin bir kısmı komşu merkezlere yönlendirilmelidir."* | (a) Merkezler arası fark 5 üzerinden 0,08 puan. (b) Maslak'ın işlerinin %34'ü arıza onarım (diğerlerinde %14–16) ve arıza onarım en düşük puan alan iş. (c) Aynı iş türleri karşılaştırıldığında Maslak dört kategorinin dördünde de en yüksek; iş karışımına göre düzeltilince beklenenin en çok üstündeki merkez (+0,054). (d) Daha fazla basamakla en altta Maslak değil İzmir1 (3,906 / 3,911). | Merkezler iş türüne göre karşılaştırılmalı. Öneri, en iyi merkezlerden birini cezalandırıp işini daha düşük puan alan merkezlere aktarmak anlamına geliyor. Maslak için aksiyon gerekmiyor; düzeltilmiş karşılaştırmada en geride İzmir1. |
| 6 | *"p = 0,03... hipotezimizin %97 ihtimalle doğru olduğunu söyleyebiliriz... Memnuniyet puanını 0,3 puan artırırsak terk oranını yaklaşık %8 azaltabiliriz."* | İki hata: (a) p-değeri "hipotezin doğru olma olasılığı" değildir; fark yokken bu büyüklükte bir fark görme olasılığıdır. (b) İlişki neden-sonuç değildir; "0,3 puan → %8" için hiçbir dayanak gösterilmemiş. | "Memnuniyeti düşük müşterilerin terk oranı belirgin şekilde yüksek" denebilir. Memnuniyeti artırmanın terki ne kadar azaltacağı ancak bir deneyle (kontrol grubu) ölçülebilir. |

### Ek hatalar

- **"Üç ana tabloyu birleştirdik... 32.770 satırlık analiz tablosu... müşteri seviyesinde analiz ettik."** Vaka dört dosyanın birleştirilmesini istiyor. 32.770 servis tablosunun ham satır sayısı; kopyalar yüzünden birleştirme 33.047 satır üretiyor. Müşteri seviyesinde tablo ~4.000 satır olmalı.
- **Memnuniyet boşluklarını ortalamayla doldurmak.** Anket zorunlu değil ve boşluklar rastgele değil: Arıza onarımda cevapsızlık %26, periyodik bakımda %19. Ortalamayla doldurmak bu bilgiyi siler ve farkları bastırır.
- **"Uç değerler analiz sonuçlarını etkilemediği için olduğu gibi bırakılmıştır."** Tutarda 2,9 milyon TL'ye varan 43 hatalı kayıt ve 150 negatif tutar var; temizlik sonrası standart sapma 37.232'den 7.852'ye düşüyor. "Etkilemedi" iddiası kontrol edilmemiş.
- **"Model çıktısı, terk riski yüksek müşterilere otomatik kampanya göndermek için kullanılabilir."** Sızıntılı bir model üretimde çalışmaz. Ayrıca LLM ile otomatik öneri üretmek, şirketin sunmadığı kampanyalara yol açabilir (Bölüm 3.3'te kısıtsız prompt "%10 indirimli paket" ve "indirim kuponu" uydurdu). İnsan onayı olmadan uygulanmamalı.